# 5.4 The answer: a ratio that turns

[05.3](05.3-covid-modelling.ipynb) left a straight line in positive tests with a residual
that goes negative after January 2021 and never comes back. This notebook is the worked
answer, and the answer is small: **the same line, multiplied by a switch.**

    deaths_shifted ≈ (a · positivetests + b) · logistic(day, k, x0)

The line is the claim that deaths are a fixed fraction of positive tests. The switch is the
claim that the fraction *turns* — one value, a transition over weeks, a smaller value. Two
parameters describe the switch: `x0`, the day on which the turn is halfway, and `k`, how
steep it is, negative so that it runs from 1 down towards 0. Four parameters in total, and
every one of them means something you can read off afterwards.

Multiplied, not added. A logistic *added* to the line would say "from some day on, a fixed
number of deaths a day disappears, however many people test positive". That is not the
claim. A protected population changes the *fraction* of positive tests that end in death,
and a fraction acts on the count: the more tests there are, the more deaths the switch
takes away. A ratio is a multiplication, so the term is multiplied in. Writing that down
correctly is the whole of the modelling; the rest is `train_model`.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.analytics import DistributionFitter, fit_table
from goad_toolkit.config import DataConfig, FileConfig
from goad_toolkit.dataprocessor import CovidDataProcessor
from goad_toolkit.models import linear_model, logistic, mse, train_model
from goad_toolkit.visualizer import (
    ComparePlot,
    ComparePlotDate,
    FitPlotSettings,
    PlotFits,
    PlotSettings,
    ResidualPlot,
)
from wa_analyzer.data import load_showcase

from scripts.covid_pipeline import fit_model, plot_residual, preprocess

## 5.4.1 The switch, and where to start it

`goad_toolkit.models.logistic(x, k, x0)` is flat, then a turn, then flat again. `x0`
places the turn and `k` sets how fast it happens; `limit` stays at 1, because the switch
scales the line rather than replacing it.

In [ ]:
t = np.linspace(-10, 10, 200)
shapes = pd.DataFrame({"t": t, "k=1: up": logistic(t, k=1, x0=0), "k=-0.5: down, slower": logistic(t, k=-0.5, x0=0)})
fig, ax = ComparePlot(PlotSettings(figsize=(7, 3.2), xlabel="t", ylabel="", title="A logistic switch")).plot(  # ty: ignore[invalid-argument-type]
    data=shapes, x="t", y1="k=1: up", y2="k=-0.5: down, slower",
)

The data is the frame 05.3 built. The line is fitted on the pre-vaccination window first,
for the same reason as there — that is the window where the ratio holds — and because its
`a` and `b` are the starting values the four-parameter fit needs. The other two starting
values come from what you already know: `k` gently negative, `x0` a few weeks after the
first vaccinations. An optimiser started nowhere near the answer will happily settle on a
different one. The bounds say what each parameter is allowed to mean: a ratio between 0
and 1, an intercept that is not negative, a switch that turns *down*, a halfway day inside
the window.

In [ ]:
VACCINATION_START = "2021-01-06"
data = CovidDataProcessor(FileConfig(), DataConfig()).process()
before = data.index < VACCINATION_START

tests = data["positivetests"].to_numpy()
deaths = data["deaths_shifted"].to_numpy()
day = np.arange(len(data)).astype(float)
X = np.stack([tests, day], axis=1)

line = train_model(tests[before], deaths[before], linear_model, mse, [0.01, 1.0],
                   bounds=[(0, 1.0), (0, None)])
data["line"] = linear_model(tests, line)


def covid_model(X: np.ndarray, params: list[float]) -> np.ndarray:
    """A straight line in positive tests, times a logistic switch on the day."""
    a, b, k, x0 = params
    return linear_model(X[:, 0], [a, b]) * logistic(X[:, 1], k=k, x0=x0)


vaccination_day = float(np.argmax(data.index >= VACCINATION_START))
initial = [line[0], line[1], -0.1, vaccination_day + 30]
params = train_model(X, deaths, covid_model, mse, initial,
                     bounds=[(0, 1.0), (0, None), (-1.0, 0), (0, len(data))])
a, b, k, x0 = params

data["turning"] = covid_model(X, params)
data["residual"] = data["deaths_shifted"] - data["turning"]
halfway = data.index[int(round(x0))].date()
print(f"the line alone:  deaths ≈ {line[0]:.4f} · tests + {line[1]:.1f}")
print(f"with the switch: a = {a:.4f}, b = {b:.1f}, k = {k:.3f}, x0 = day {x0:.0f} = {halfway}")
print(f"mse: the line on the full window {mse(deaths, data['line']):.0f}  →  the line times the switch {mse(deaths, data['turning']):.0f}")

In [ ]:
fig, ax = ComparePlotDate(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                       title="Deaths, and a model whose ratio turns")).plot(
    data=data, x="date", y1="deaths_shifted", y2="turning",
    date=VACCINATION_START, datelabel="vaccination starts",
)

## 5.4.2 Reading the parameters

`x0` puts the halfway point of the turn in **mid-March 2021**, about ten weeks after the
first vaccinations. `k` is small, which means the turn is gradual: the switch runs from
about four fifths to about one fifth over the two months centred on that day, and by
May the ratio is little more than a tenth of its winter value.

Read `a` and `b` with more care. They are not the pre-vaccination line's numbers — the
slope came out lower and the intercept higher — because a switch this gradual is already
slightly below 1 in January and February, and the line adjusts to share the work with it.
Four parameters fitted together are not four separate findings; reading any one of them
alone is a mistake. What the model claims on a given day is their *product*, the ratio of
deaths to tests it implies, and that is the number to quote.

Set that against the rollout. The Dutch programme started on 6 January with care-home
residents and the people who look after them, and worked down the age groups from the
oldest through February and March — which is to say it reached the people who carried
most of the deaths well before it reached most of the population. A turn in the
deaths-per-test ratio that is halfway by mid-March, while overall coverage was still low,
is what that ordering predicts. The parameters are a finding: they date the turn, and the
date is consistent with the mechanism you had in mind.

Consistent with, not proof of. The model has a switch on the calendar, not a term for
vaccination. The end of the winter wave, a change in who was being tested, and the order
in which age groups were vaccinated all sit on the same calendar, and each of them would
produce a switch that fits. Fitting the residual showed that the ratio turned, and when.
**Why it turned is a separate claim** — [05.1](05.1-relationships.ipynb)'s mechanism leg,
which no fit supplies — and defending it needs something other than this model:
vaccination coverage by age group set against deaths by age group, say, or the same fit on
a country whose rollout started on a different date.

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="The residual, once the ratio is allowed to turn")).plot(
    data=data, x="date", y="residual", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

In [ ]:
fitter = DistributionFitter(seed=42)
fits = fitter.fit(data["residual"].to_numpy(), discrete=False)
print(fit_table(fits)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4).to_string())

fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Full window, with the switch: the residual, top three fits")).plot(
    data=data["residual"].to_numpy(), fit_results=fits,
    fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

The long negative run is gone, the error is a tenth of what the line left behind, and
the residual is back to what the pre-vaccination window looked like: symmetric, several
families within a few log-likelihood points of each other, none rejected. By the bar 05.3
set — no structure in time, a plausible family — this model is done. Not perfect: the
weekly waves are still there, and a serious model of this would treat them (a sine with
period seven, or the reporting cycle taken out in the pipeline). But nothing left in the
residual has a shape that names a mechanism the model lacks.

Two things are worth noticing about how small the fix was. It added one term and two
parameters, and it did not touch the line. And the term was chosen by asking what the data
means — a fraction of tests, changing over weeks — rather than by trying shapes until one
fitted. That is the difference between a model you can defend and one that merely matches.

## 5.4.3 One more basis function

The switch is one of a small family of shapes worth knowing by name — **linear, sine,
exponential, logistic** — and the habit is the same for each: look at what the current
model cannot explain, name the shape of what is left, and add that shape as a term. A line
says "proportional". An exponential says "compounding". A logistic says "it turns". A sine
says "it repeats". Monthly airline passengers, 1949–1960, is the plainest case of the last
one: a trend, and a cycle that comes back every twelve months.

In [ ]:
flights = load_showcase("flights")
months = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"], format="%Y-%b")
t = np.arange(len(flights)).astype(float)
y = flights["passengers"].to_numpy().astype(float)


def trend_plus_seasonal(t: np.ndarray, params: list[float]) -> np.ndarray:
    a, b, amplitude, phase = params
    return a * t + b + amplitude * np.sin(2 * np.pi * t / 12 + phase)


trend_only = train_model(t, y, linear_model, mse, [1.0, 100.0])
trend_and_cycle = train_model(t, y, trend_plus_seasonal, mse, [1.0, 100.0, 30.0, 0.0])
compare = pd.DataFrame({
    "month": months,
    "actual": y,
    "trend only": linear_model(t, trend_only),
    "trend + 12-month sine": trend_plus_seasonal(t, trend_and_cycle),
})
for column in ["trend only", "trend + 12-month sine"]:
    explained = 1 - np.var(y - compare[column]) / np.var(y)
    print(f"{column:22s} variance explained {explained:.1%}")

settings = PlotSettings(figsize=(10, 4.5), title="Airline passengers: a line, and a line plus a sine",  # ty: ignore[invalid-argument-type]
                        xlabel="", ylabel="passengers")
fig, ax = ComparePlot(settings).plot(data=compare, x="month", y1="trend only", y2="trend + 12-month sine")
ax.scatter(compare["month"], compare["actual"], s=8, color="black", alpha=0.5, label="actual", zorder=3)
_ = ax.legend()

One added term — the amplitude and phase of a 12-month sine, four parameters in total —
lifts the explained variance from 85% to 93%. The gap that remains is visible and honest:
the real cycle *widens* as the trend grows, and a sine added on top of a line stays the
same width throughout. Getting the rest would mean multiplying the cycle by the trend
instead of adding it — the same added-or-multiplied decision the covid switch turned on,
and the natural next term rather than a failure of this one.

## 5.4.4 Why this lives in a script

The loop above — process, compare, model, residual, distribution fit — is also
`scripts/covid_pipeline.py`: six functions and a `main()` that runs them in order. Open it
next to this notebook. Each function has a name a reader can guess the contents of, and a
signature that says what goes in and what comes out:

| function | in | out | the decision it owns |
|---|---|---|---|
| `preprocess()` | nothing (config) | the processed frame | which pipeline steps make the data |
| `plot_zscores(data)` | the frame | a figure | whether the two series share a shape |
| `fit_model(data)` | the frame | the frame, plus `predicted deaths` and `residual` | the model, its starting values, its bounds |
| `plot_model(data)` | the fitted frame | a figure | what the fit looks like against the data |
| `plot_residual(data)` | the fitted frame | a figure | whether the error has structure over time |
| `plot_residual_distribution(data)` | the fitted frame | a figure and a `fit_table` | whether the error is noise |

None of them is more than twenty lines, and none of them knows about the others:
`fit_model` returns a *new* frame rather than editing the one it was given, so running it
twice does not double-fit, and every plotting function takes the frame it needs as an
argument rather than reaching for a global. A notebook cell has no such boundary — nothing
stops one cell's variable from leaking into the next, and nothing forces the logic small
enough to test.

Three things the script can do that the cells above cannot: re-run one step (change the
model in `fit_model`, and nothing upstream has to run again); be imported — the next cell
does exactly that, and a test could call `fit_model` on ten made-up rows; and run on a
schedule, `uv run python scripts/covid_pipeline.py`, with no kernel, no cell order, and no
human. The cost is that a function has to *say* what it needs. That constraint is the
feature.

In [ ]:
scripted = fit_model(preprocess())
fig, ax = plot_residual(scripted)

The same fit and the same residual as §5.4.2, in two calls; the log lines above them are
the script talking — what it loaded, what it fitted, where it saved the figure.

## What to carry forward

1. **The residual is the finding, not the leftover.** The line was improved by asking what
   the *error* still looked like, not by staring harder at the fit; the shape of the error
   named the term, and the day it began dated it.
2. **A basis function is a claim about the shape you expect.** A logistic said "the ratio
   turns". A sine said "this repeats every twelve months". Both are testable, and both were
   confirmed by the residual shrinking once they were added — not assumed because they
   seemed reasonable. How the term combines with what is already there is part of the claim.
3. **The model dates the turn; the mechanism is yours to defend.** A switch on the calendar
   is consistent with vaccination, and with everything else that happened on that calendar.
4. **A script is a notebook with the leakage removed.** Small functions with explicit
   inputs are what let you re-run one step, import it elsewhere, or test it without
   re-running everything above it.